# AutoSDV HPC tracing 분석

본 노트북은 `run_trace_session.sh`로 수집한 4-config × 3 runs trace를 읽고
T_HPC 분포 / per-callback duration / run-to-run variance 를 산출한다.

**선행 조건 (HPC에서):**
```
sudo apt install -y ros-humble-tracetools-analysis python3-bt2
pip install pandas matplotlib seaborn pyarrow
```

**입력:**
- 세션 디렉토리: `/tmp/autosdv_traces/autosdv_hpc_<config>_run<NN>/`
  - `ust/...` (LTTng CTF trace)
  - `csv/entry_*.csv`, `csv/exit_*.csv`

**산출 (plan §6.3):**
- T_HPC CDF per config
- per-callback duration box plot
- Run-to-run mean ± std bar chart


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# frame_id_join.py를 모듈로 import (같은 디렉토리에 있다고 가정)
sys.path.insert(0, str(Path.cwd()))
import frame_id_join as fij

TRACE_BASE = Path('/tmp/autosdv_traces')
CONFIGS = ['S1', 'S2', 'S3', 'S4']
NUM_RUNS = 3

## 1. 모든 run에 대해 T_HPC 계산

In [ ]:
def session_dir(cfg: str, run: int) -> Path:
    return TRACE_BASE / f'autosdv_hpc_{cfg}_run{run:02d}'

all_results = []
for cfg in CONFIGS:
    for run in range(1, NUM_RUNS + 1):
        sd = session_dir(cfg, run)
        if not sd.exists():
            print(f'  skip (missing): {sd}')
            continue
        try:
            csv_df = fij.load_application_csv(sd)
            trace_df = fij.load_trace_events(sd)
            joined = fij.join_frame_id(csv_df, trace_df)
            t_hpc = fij.compute_t_hpc(joined)
        except Exception as e:
            print(f'  ERROR {cfg} run{run}: {e}')
            continue

        # warm-up 30s discard
        if len(t_hpc):
            t0 = t_hpc['in_ts_ns'].min()
            t_hpc = t_hpc[t_hpc['in_ts_ns'] >= t0 + 30 * 1_000_000_000]

        t_hpc['config'] = cfg
        t_hpc['run'] = run
        all_results.append(t_hpc)
        print(f'  {cfg} run{run}: {len(t_hpc)} frames, '
              f'mean={t_hpc["t_hpc_ms"].mean():.2f} ms')

df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
df.head()

## 2. T_HPC CDF per config

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for cfg in CONFIGS:
    vals = df.loc[df['config'] == cfg, 't_hpc_ms'].sort_values().values
    if len(vals) == 0:
        continue
    cdf = np.arange(1, len(vals) + 1) / len(vals)
    ax.plot(vals, cdf, label=cfg)
ax.set_xlabel('T_HPC (ms)')
ax.set_ylabel('CDF')
ax.set_title('HPC 내부 처리 지연 분포 (3 runs aggregated)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig('t_hpc_cdf.png', dpi=120)
plt.show()

## 3. Run-to-run variance (mean ± std)

In [ ]:
per_run = df.groupby(['config', 'run'])['t_hpc_ms'].mean().reset_index()
summary = per_run.groupby('config')['t_hpc_ms'].agg(['mean', 'std']).reset_index()
print(summary)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(summary['config'], summary['mean'], yerr=summary['std'],
       capsize=8, alpha=0.8)
ax.set_ylabel('T_HPC mean across runs (ms)')
ax.set_title('Run-to-run variance (n=3 per config)')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('t_hpc_variance.png', dpi=120)
plt.show()

## 4. tracetools_analysis로 callback duration

ROS2 공식 분석 라이브러리. HPC에 설치된 경우만 동작.

In [ ]:
try:
    from tracetools_analysis.loading import load_file
    from tracetools_analysis.processor import Processor
    from tracetools_analysis.processor.ros2 import Ros2Handler
    from tracetools_analysis.data_model.ros2 import Ros2DataModel
    HAVE_TA = True
except ImportError as e:
    print(f'tracetools_analysis 미설치: {e}')
    HAVE_TA = False

In [ ]:
if HAVE_TA:
    sd = session_dir('S1', 1)  # 대표 1개
    events = load_file(str(sd / 'ust'))
    handler = Ros2Handler.process(events)
    model: Ros2DataModel = handler.data

    # callback_instances DataFrame
    cb = model.callback_instances.copy()
    cb['duration_ms'] = (cb['timestamp'].diff().shift(-1).fillna(0) * 1e-6)  # 데모용 근사
    print(cb.head())
else:
    print('tracetools_analysis가 없어 callback 분석 스킵.')

## 5. 통계 검정 (옵션) — TAS vs BE 차이 유의성

In [ ]:
from scipy import stats as scs

be = df.loc[df['config'] == 'S1', 't_hpc_ms']
tas = df.loc[df['config'] == 'S2', 't_hpc_ms']
if len(be) and len(tas):
    u, p = scs.mannwhitneyu(be, tas, alternative='two-sided')
    print(f'Mann-Whitney U (S1 vs S2): U={u:.2f}, p={p:.4g}')
    print(f'  → p<0.05 면 분포 차이 유의')
else:
    print('데이터 부족')